# 05 — Évaluation et généralisation

- Détection : Precision, Recall, F1, mAP@50, mAP@50-95.
- Segmentation : IoU, Dice, mAP de segmentation.
- Cartographie : taux de confirmation, faux signalements par parcours.

**Split strict par parcours** : aucune image d'un même parcours dans train et test.

In [ ]:
import sys
sys.path.append("../evaluation")
from metrics import evaluate_by_route, precision_recall_f1, dice

routes = {
    "parcours-yopougon-01": (34, 5, 6),
    "parcours-abobo-02": (28, 4, 9),
    "parcours-adjame-03": (41, 3, 7),
}
evaluate_by_route(routes)

In [ ]:
import json
from pathlib import Path

# Courbes mAP par époque, exportées par Ultralytics dans runs/detect/*/results.csv
import pandas as pd
csv = list(Path("..").glob("**/runs/**/results.csv"))
if csv:
    df = pd.read_csv(csv[0])
    df.columns = [c.strip() for c in df.columns]
    df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].plot(x="epoch")

In [ ]:
# Cartographie : recharger la base et recalculer les hotspots
import sys
sys.path.append("../backend")
from app.geolocation import detect_hotspots, merge_session_points
from app.models import SessionLocal, Observation

db = SessionLocal()
rows = [{"id": o.id, "session_id": o.session_id, "captured_at": o.captured_at.timestamp(),
         "latitude": o.latitude, "longitude": o.longitude,
         "zone_category": o.zone_category, "validation_status": o.validation_status}
        for o in db.query(Observation).all()]
hotspots = detect_hotspots(merge_session_points(rows))
hotspots[:5]